# Step 2 - Prototype 1

Here we mainly want to focus on adding the option to have trains going in both directions. In this context, we might have to implement dwell times (to handle buffering) and decide if tracks (all or only some) are directed or undirected. Additionally, we might have to rework the train schedules (more specifically their formulation) to some extent. All in all, this will set us up for longer time horizons.

Also, we might introduce train and station (passenger) weights as more realistic constraints here could potentially solve (or at least mitigate) lazy solver behavior. This could then allow us to get rid of the 'reach station by the end of the time horizon constraint', which might be somewhat problematic in the context of longer time horizons and train schedules with back and forth plans.

To get started, we use the code from the cleaned file for step 1 and iteratively modify it.

In [60]:
# Import Gurobi
import gurobipy as gp
from gurobipy import GRB

import numpy as np

In [61]:
# Initialize the Model
model = gp.Model("Basic_MILP")

### Adjusting the tack blueprint information

Here we also might make some modifications (like maximum allowed speed) but probably also not in this prototype...

In [62]:
# Define the track system using dictionaries to have homogeneous lists and allow for precise track building

# Copy preset for faster building
# {"type": "station", "capacity": 0}
# {"type": "segment", "length": 0, "capacity": 0}

track_blueprint = [
    {"type": "station", "capacity": 3},
    {"type": "segment", "length": 5, "capacity": 1},
    {"type": "station", "capacity": 2},
    {"type": "segment", "length": 2, "capacity": 2},
    {"type": "station", "capacity": 3}
]

In [63]:
# Parsing the track blueprint and generating necessary data (1D lists for Gurobi)
# These store the allowed number of trains on a given block and which blocks are train stations

block_capacities = []
is_station = []

for item in track_blueprint:
    if item["type"] == "station":
        block_capacities.append(item["capacity"])
        is_station.append(True)
    elif item["type"] == "segment":
        # Int conversion not necessary, but gets rid of IDE highlighting
        for _ in range(int(item["length"])):
            block_capacities.append(item["capacity"])
            is_station.append(False)
    else:
        print("typo? No other types yet...")

In [64]:
np.random.randint(0, 3)

2

### Adjusting the train schedules

Here we want to add additional train information (weights related to the number of passengers). Also, but probably not in this prototype, we want to add additional specifications like maximum speed.

**Another idea (possibly for later):** We can add random noise to the stations like

`{"station": 0, "departure": 3, "random_delay": np.random.rand(np.random.randint(0, 3))}`

to our stations and have additional constraints enforcing those to simulate a more dynamic and realistic network. The same could be done for tracks (switches). Generally, in a similar fashion (but maybe without randomness) we could introduce delays or issues into the system and see how the solver adapts the optimal solution then. 

In [65]:
# Similar to our track_blueprint, we use dictionaries for the schedules for a given train
# Especially here, this allows for basically arbitrary additional data that can be used for
# different optimizing goals and constraints. This also nicely handles, that a train doesn't
# have to stop at every station.

# Note: Due to the implementation ('looking-back') departure_times_i has to be >= 1
# Thought: Should we have an (optional) boolean 'is_destination' in (some) station
# descriptions or will this always be clear? Maybe then we could also extract this information
# using a parsing function to create lists / a matrix is_destination...
train_information = [
    # Train 0 (RE-like)
    {
        # Info can be essentially arbitrary and will likely just be used for better optimization goals
        "info": {"passenger_count": 100},
        "schedule": [
            # First journey
            {"station": 0, "departure": 3},
            {"station": 1, "arrival": 10, "departure": 13},
            # (Partially xD) Second journey
            {"station": 2, "arrival": 12, "departure": 20}, # Merged arrival / departure for the turnaround
            {"station": 1, "arrival": 27, "departure": 29},
            {"station": 0, "arrival": 34}
        ]
    },
    # Train 1 (ICE-like)
    {
        "info": {"passenger_count": 250},
        "schedule": [
            # First journey
            {"station": 0, "departure": 5},
            # (Partially xD) Second journey
            {"station": 2, "arrival": 13, "departure": 18, "dwell": 5}, # Merged arrival / departure for the turnaround
            {"station": 0, "arrival": 29}
        ]
    },
    # More trains ...
]

In [66]:
# Number of time steps over which we optimize
time_horizon = 50

# Deriving some additional variables for bounding variables
num_trains = len(train_information)
num_blocks = len(block_capacities)
num_stations = sum(is_station)

# List of all block indices that are stations
station_block_indices = [block for block, station_idx in enumerate(is_station) if station_idx]

### Adjusting the decision variable(s)

Instead of only having one 3D-Tensor, we will now have two (one for the forward and one for the backward direction). Using this formulation we will have to adjust most of our constraints. However, this should rather just be considering both tensors instead of actual logical changes.

In [67]:
# Define Decision Variables

# To describe our train movement we need a way to know which train is where at what
# time (-> 3D time-position grid of boolean values)

x_fwd = model.addVars(num_trains, num_blocks, time_horizon, vtype=GRB.BINARY, name="x_fwd")
x_bwd = model.addVars(num_trains, num_blocks, time_horizon, vtype=GRB.BINARY, name="x_bwd")

### Constraints (Update this)

#### General Constraints

As general constraints, we have the
- uniqueness constraint (no train can be on multiple tracks at once)
- spawning constraint (a train does not 'exist' until it leaves from the first station in its schedule). For future iterations we should also think about if we want to keep the spawning behavior or replace (or even remove?) it...
- capacity constraints (there can't exist more trains on a block (or station) than its capacity)

#### Movement Constraints

For now, we have the
- 'at most one block per timestep' constraint (will likely be changed in the next phase (step 2) to allow for different train speeds)
- 'departure-time constraint' (trains don't have to stop at stations if they are not on their schedule and also, they can't leave a station until their departure time)

In [68]:
# Testing cell for dictionary access again
i = 0
train_information[i]["schedule"][0]["departure"]

3

In [69]:
# Uniqueness constraint for trains to only exist once on a block (on only a single track at a given time)
# Additionally, we tackle the spawning (departure times), as before a train does not exist on any block
for i in range(num_trains):

    # Extract the spawn time (first departure time)
    spawn_time = train_information[i]["schedule"][0]["departure"]

    for k in range(time_horizon):

        # Sum over all blocks j to ensure that a train only exists once at a given time k
        active_tracks = gp.quicksum(x_fwd[i, j, k] + x_bwd[i, j, k] for j in range(num_blocks))

        # If the train hasn't spawned (or left) yet, it can't exist on the track and we enforce this
        if k < spawn_time:
            model.addConstr(active_tracks == 0, name=f"NotSpawned_Train{i}_Time{k}")
        else:
            # If it already exists, we enforce that the train only exists on one track at any time k
            model.addConstr(active_tracks == 1, name=f"UniquePosition_Train{i}_Time{k}")

In [70]:
# Capacity constraints (the sum over all tracks j and all trains i <= capacities_i)
for k in range(time_horizon):

    # Here we want to sum over all i AND j and add the constraint that this should be smaller than
    # Is this formulation correct and can this be made more compact (yes: x.sum('*', j, k))?
    for j in range(num_blocks):
        occupied_tracks = gp.quicksum(x_fwd[i, j, k] + x_bwd[i, j, k] for i in range(num_trains))

        # Add constraint to enforce capacity maximum across block j
        model.addConstr(occupied_tracks <= block_capacities[j], name=f"Capacity_Block{j}_Time{k}")

In [71]:
# Movement Constraints

# A train can (after spawning) either wait or move
for i in range(num_trains):

    # Extract the spawn station id and spawn time (first departure time)
    spawn_station_id = train_information[i]["schedule"][0]["station"]
    spawn_time = train_information[i]["schedule"][0]["departure"]

    # At every time point, we need to differentiate between different scenarios
    for k in range(time_horizon):

        # If the train hasn't left yet, it's not yet on the grid and the spawn constraint takes care of this
        if k < spawn_time:
            pass
        # Here the train spawns, and we fix a position (the block that corresponds to the first station on
        # the trains schedule)
        elif k == spawn_time:

            # Here we can again use the spawn_station_id and get the one for the next station
            block_index = station_block_indices[spawn_station_id]
            next_station_id = train_information[i]["schedule"][1]["station"]

            # Determine the initial direction of the train
            if spawn_station_id < next_station_id:
                # Moving right -> Spawn in Forward tensor
                model.addConstr(x_fwd[i, block_index, k] == 1, name=f"SpawnFwd_Train{i}_Time{k}_Block{block_index}")
            else:
                # Moving left -> Spawn in Backward tensor
                model.addConstr(x_bwd[i, block_index, k] == 1, name=f"SpawnBwd_Train{i}_Time{k}_Block{block_index}")

        else:

            # Handle out back-looking approach for both channels (tensors)
            for j in range(num_blocks):

                # Forward channel (move 'right')
                # If the train is on block one, it must have already been there (j is always ≠ -1)
                if j == 0:
                    model.addConstr(x_fwd[i, 0, k] <= x_fwd[i, 0, k - 1], name=f"MoveFwd_Train{i}_Time{k}_Block0")
                else:
                    # General back-look constraint (train must have been here or in the previous block)
                    model.addConstr(x_fwd[i, j, k] <= x_fwd[i, j, k - 1] + x_fwd[i, j - 1, k - 1], name=f"MoveFwd_Train{i}_Time{k}_Block{j}")

                # Backward channel (move 'left')
                # If the train is on block one, it must have already been there (j is always ≠ -1)
                if j == num_blocks - 1:
                    model.addConstr(x_bwd[i, 0, k] <= x_bwd[i, 0, k - 1], name=f"MoveBwd_Train{i}_Time{k}_Block0")
                else:
                    # General back-look constraint (train must have been here or in the previous block)
                    model.addConstr(x_bwd[i, j, k] <= x_bwd[i, j, k - 1] + x_bwd[i, j + 1, k - 1], name=f"Move_Train{i}_Time{k}_Block{j}")

### Adjusting intermediate (and destination) station constraints and handling

Here we might need / want the information available if a station is a destination station. Also, here we probably need to still do some more modifications to the function and the logic...

In [ ]:
# Now, we want to add the constraints for intermediate stations (arrival and departure times and no spawning)
# In this version we only tackle departure times and not dwell times yet

# For every train
for i in range(num_trains):

    # Get the spawn station ID so we can skip it (this is already handled by the spawning constraint)
    spawn_station_id = train_information[i]["schedule"][0]["station"]

    # Here we have to change the loop / iteration logic since we have a list of dictionaries now...
    # Iterate directly through this train's schedule
    for station_id, schedule in train_information[i]["schedule"].items():

        # Skip the spawn station (already handled)
        if station_id == spawn_station_id:
            continue

        # Check if this station has a strict leave time (might always be the case, but we'll see)
        if "departure" in schedule:
            leave_time = schedule["departure"]
            station_block = station_block_indices[station_id]
            next_block = station_block + 1

            # Ensure we don't look past the end of the tracks (should only be an issue with the last station)
            if next_block < num_blocks:

                # Build the wall: The train cannot enter the next block before the leave time
                for k in range(leave_time):
                    model.addConstr(
                        x[i, next_block, k] == 0,
                        name=f"DepartureWall_Train{i}_Block{next_block}_Time{k}"
                    )
            else:
                # Here we might have some code later when having more complex schedules and a train e.g.,
                # 'turns around' and continues its journey back to the initial station. However, this
                # might then also be handled differently again...
                pass